# 7장 — DPO: 보상 모델 없이 조련하기 ✂️

6장 노트북과 결정적으로 다른 점 하나를 먼저 보세요 — **이 노트북에는 `import random` 이 없습니다.**

RLHF는 샘플링(생성)→채점 루프가 필요했지만, DPO는 선호 쌍 데이터에 대해 손실을 재고
내리기만 하면 됩니다(지도학습 스타일). 흐름:
1. 숨은 점수 r̂ = β·log(π/π_ref) 와 DPO 손실 −log σ(마진) 구현
2. **수치미분**으로 기울기를 재서(공식 유도 없이!) 경사하강
3. 선택된 답은 원본보다 자주, 탈락한 답은 덜 자주 — 격차가 벌어지는 것을 확인

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 오른쪽 아래 버튼 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch7')

## 1. 숨은 점수와 DPO 손실

- 버킷 3개: W(선택된 답) · L(탈락한 답) · 기타(그 외 모든 답).
- 정책은 로짓 z 로 조종: π ∝ π_ref · e^z — z가 전부 0이면 정책 = 참조 모델(시작점).
- 숨은 점수 r̂ = β·log(π/π_ref), 손실 = −log σ(r̂_W − r̂_L).

> 여기서 log 는 **자연로그 ln**(논문 표준 — 아래 코드의 `math.log`도 자연로그).
> 감 잡기: β=0.1, π=0.2, π_ref=0.1 이면 비율 2 → r̂ = 0.1 × ln 2 ≈ **+0.069** —
> 원본보다 2배 자주 내는 답이라 점수가 양수입니다.

In [ ]:
import math

NAMES = ['W 선택된 답', 'L 탈락한 답', '기타 답변들']
PI_REF = [0.30, 0.30, 0.40]  # 참조 모델(SFT)의 분포 — 고정
BETA = 1.0

def policy(z):
    w = [PI_REF[i] * math.exp(z[i]) for i in range(3)]
    s = sum(w)
    return [x / s for x in w]

def sigmoid(x):
    return 1 / (1 + math.exp(-x))

def dpo_loss(z):
    pi = policy(z)
    r_w = BETA * math.log(pi[0] / PI_REF[0])  # 숨은 점수 (선택)
    r_l = BETA * math.log(pi[1] / PI_REF[1])  # 숨은 점수 (탈락)
    return -math.log2(sigmoid(r_w - r_l))

z = [0.0, 0.0, 0.0]
print(f'시작: 정책 = 참조 모델 {[f"{p:.0%}" for p in policy(z)]}')
print(f'시작 손실 = {dpo_loss(z):.2f}  (마진 0 → σ(0)=0.5 → −log₂(0.5) = 1)')

## 2. 수치미분으로 경사하강 — 공식 없이 기울기 재기

2부에서 배운 미분의 정의를 그대로 씁니다: 로짓을 아주 조금(h) 흔들어 보고
손실이 얼마나 변하는지 재면 그게 기울기예요. 공식 유도 없이도 훈련이 됩니다
(1부 캡스톤에서 '수치미분 검산'으로 만난 그 기법).

In [ ]:
H, LR = 1e-5, 0.4

def numeric_grad(z):
    """기울기 = (조금 흔든 손실 − 원래 손실) / h  — 각 로짓마다."""
    base = dpo_loss(z)
    grads = []
    for i in range(3):
        z2 = z.copy()
        z2[i] += H
        grads.append((dpo_loss(z2) - base) / H)
    return grads

# P(사람 취향 적중) 90% 에서 멈춘다 — 계속 밀어붙이면 격차 벌리기에만 몰두해
# 비교 안 된 분포(원본다움)까지 쓸려간다 (β 를 느슨하게 푼 것과 같은 효과, 6장의 교훈)
TARGET_LOSS = -math.log2(0.9)
curve = [dpo_loss(z)]
while curve[-1] > TARGET_LOSS and len(curve) < 200:
    g = numeric_grad(z)
    z = [z[i] - LR * g[i] for i in range(3)]  # 손실이 주는 방향으로 (경사하강)
    curve.append(dpo_loss(z))

print(f'조기 종료: {len(curve) - 1} 스텝 (P ≥ 90% 도달)')
pi = policy(z)
print('훈련 후 정책 vs 참조:')
for i in range(3):
    arrow = '↑' if pi[i] > PI_REF[i] + 0.01 else ('↓' if pi[i] < PI_REF[i] - 0.01 else '≈')
    print(f'   {NAMES[i]:10s} π_ref {PI_REF[i]:.0%} → π {pi[i]:.0%}  {arrow}')
print(f'손실: {curve[0]:.2f} → {curve[-1]:.2f}')

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 3))
plt.plot(curve, marker='o', markersize=3, color='#6366f1')
plt.xlabel('훈련 스텝')
plt.ylabel('DPO 손실')
plt.title('DPO — 생성도 채점도 없이 내려가는 손실')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. 검증 — 지름길이 약속을 지켰나

In [ ]:
r_w = BETA * math.log(pi[0] / PI_REF[0])
r_l = BETA * math.log(pi[1] / PI_REF[1])
print(f'숨은 점수: W {r_w:+.2f} / L {r_l:+.2f} → 마진 {r_w - r_l:+.2f}')
print(f'P(사람 취향 적중) = σ(마진) = {sigmoid(r_w - r_l):.0%}')

# 검증 1 — 손실이 크게 내려갔다 (1비트 → 0.15비트 근처)
assert curve[-1] < curve[0] * 0.5, 'DPO 훈련은 손실을 크게 낮춰야 한다!'
# 검증 2 — 선택은 원본보다 자주, 탈락은 덜 자주
assert pi[0] > PI_REF[0] and pi[1] < PI_REF[1], 'W는 π_ref보다 ↑, L은 ↓ 여야 한다!'
# 검증 3 — 비교 안 된 '기타'는 상대적으로 조금만 변함 (조기 종료 덕에 원본다움 보존)
assert abs(pi[2] - PI_REF[2]) < 0.15, '비교되지 않은 답들의 분포는 크게 변하지 않아야 한다!'
print('\n✔ 확인: 격차는 벌어지고, 원본다움은 보존 — 심판 없이, 주사위 없이')

## 4. 관찰 정리 + 🤖 AI 연결

- **`import random` 이 없었다** — 생성(샘플링)도 채점도 없이, 선호 쌍에 대한 손실 하강만으로
  선호가 새겨졌습니다. DPO가 '지도학습 스타일'이라 불리는 이유.
- 숨은 점수 β·log(π/π_ref) 안에 **β와 π_ref(KL 목줄)가 그대로** — 지름길에도 목줄은 산다.
- 수치미분(2부의 정의 그대로)만으로 훈련이 됐다는 것도 기억하세요 — 진짜 구현은
  역전파(해석적 기울기)로 같은 일을 훨씬 빠르게 할 뿐입니다.
- −log σ(마진): 사전학습 → SFT → 보상 모델 → DPO, **−log(정답 확률)의 네 번째 재등장.**

**다음 챕터**: 8장 — 70억 개 가중치를 통째로 훈련하지 않고 "살짝만" 조련하는 LoRA.
1부의 rank(저차원) 감각이 돌아옵니다.

> 💡 궁금한 코드는 그 셀을 실행한 다음 셀에서 `from tutor import explain` 후 `explain('ch7')`.